# Portfolio Optimization with QAOA — a divi custom-workflow tutorial

This notebook uses Divi to study a partitioned QAOA formulation of portfolio optimization. It uses Divi's built-in community partitioner and beam-search aggregation.

The notebook runs end-to-end locally on `MaestroSimulator`.

## The problem

Portfolio optimization selects assets to **maximize return while minimizing risk**. Formulated as a QUBO:

$$\text{Minimize: } x^T \Sigma x - \lambda \mu^T x$$

where $\Sigma$ is the covariance matrix, $\mu$ is the returns vector, $x \in \{0,1\}^n$ are asset-selection bits, and $\lambda$ trades risk against return.

The small example uses a single QAOA. The larger example partitions the QUBO interaction graph into communities and solves one QAOA per community.

In [ ]:
import time

import numpy as np

# Local default — runs every QAOA on MaestroSimulator.
from divi.backends import MaestroSimulator

# This tutorial uses only the local backend.

---

## Phase 1 — Small Portfolio (Local)

Generate synthetic data for a small portfolio to prove the workflow.

In [ ]:
np.random.seed(42)
n_assets = 8

returns = np.random.uniform(0.01, 0.1, n_assets)
A = np.random.randn(n_assets, n_assets) * 0.1
covariance = A @ A.T + np.eye(n_assets) * 0.01  # positive definite

print(f"Portfolio: {n_assets} assets")
print(f"  returns range: [{returns.min():.4f}, {returns.max():.4f}]")
print(f"  covariance shape: {covariance.shape}")

In [ ]:
from utils import build_full_portfolio_qubo, evaluate_solution
import dimod

from divi.qprog import QAOA
from divi.qprog.problems import BinaryOptimizationProblem
from divi.qprog.optimizers import MonteCarloOptimizer

LAMBDA_PARAM = 0.75

qubo_matrix = build_full_portfolio_qubo(returns, covariance, lambda_param=LAMBDA_PARAM)
bqm = dimod.BinaryQuadraticModel(qubo_matrix, "BINARY")
print(f"QUBO: {len(bqm.variables)} variables")

**The divi primitives, briefly:**

- **`BinaryOptimizationProblem`** wraps a `dimod.BinaryQuadraticModel` as a divi problem. With no `decomposer` argument (Phase 1) it's solved as a single QAOA; with one (Phase 2) it gets partitioned.
- **`QAOA(...).run()`** is the single-program path. `n_layers=2` is the QAOA circuit depth (parameter $p$); `max_iterations=10` caps the classical outer loop.
- **`MonteCarloOptimizer`** is divi's sampling-based classical optimizer. `population_size=30` parameter sets per iteration, `n_best_sets=5` survivors carried forward.
- **`MaestroSimulator(shots=10_000)`** is the local backend used in this notebook.

In [ ]:
backend = MaestroSimulator(shots=10_000)

optim = MonteCarloOptimizer(population_size=30, n_best_sets=5)

print("Running QAOA...")
t0 = time.time()

qaoa = QAOA(
    problem=BinaryOptimizationProblem(bqm),
    n_layers=2,
    optimizer=optim,
    max_iterations=10,
    backend=backend,
)
qaoa.run()

local_time = time.time() - t0
print(f"QAOA complete in {local_time:.1f}s")

In [ ]:
# Extract and evaluate solution
qaoa_solution = np.array(qaoa.solution, dtype=int)
print(f"QAOA solution: {qaoa_solution}")
print(f"Assets selected: {np.where(qaoa_solution == 1)[0]}")

evaluate_solution(qubo_matrix, qaoa_solution, returns, covariance)

---

## Partitioned portfolio example (S&P 500, 484 assets)

The 8-asset toy fits in a single QAOA. For the larger example, we partition the QUBO into communities, solve each community with QAOA, and combine candidate assignments with beam-search aggregation.

`CommunityDecomposer` is Divi's built-in community-based partitioner. It plugs directly into `BinaryOptimizationProblem(decomposer=...)`.

In [ ]:
real_returns = np.load("2016-01-01_returns.npy")
real_covariance = np.load("2016-01-01_covariance.npy")
real_correlation = np.load("2016-01-01_correlation.npy")
print(f"Real portfolio: {len(real_returns)} assets (S&P 500, 2016-01-01)")

### How divi turns a custom partitioner into a workflow

The two cells below wire three things together — here's what each one is doing:

**`CommunityDecomposer(...)` — partitioning.** Divi builds communities from the QUBO interaction graph and creates one sub-program per community. `max_cluster_size` bounds the size of each sub-problem.

**`hybrid.SplatComposer()` — composition.** It places each partition's bits into a single global assignment.

**`PartitioningProgramEnsemble.aggregate_results(BeamSearchStrategy(...))`.** Each partition's QAOA returns candidate bitstrings. The strategy keeps the top candidates per partition and combines them according to the full QUBO energy.

In [ ]:
from divi.qprog import BeamSearchStrategy, EarlyStopping
from divi.qprog.problems import CommunityDecomposer
from divi.qprog.workflows import PartitioningProgramEnsemble

MAX_PARTITION_SIZE = 20  # max assets per QAOA sub-problem

real_qubo = build_full_portfolio_qubo(
    real_returns, real_covariance, lambda_param=LAMBDA_PARAM
)

problem = BinaryOptimizationProblem(
    real_qubo,
    decomposer=CommunityDecomposer(
        max_cluster_size=MAX_PARTITION_SIZE,
        method="modularity",
        seed=42,
    ),
    composer=hybrid.SplatComposer(),
)

ensemble = PartitioningProgramEnsemble(
    problem=problem,
    quantum_routine="qaoa",
    n_layers=2,
    optimizer=MonteCarloOptimizer(population_size=30, n_best_sets=5),
    max_iterations=10,
    early_stopping=EarlyStopping(patience=3),
    backend=backend,
)

print("Decomposing portfolio + creating sub-programs...")
ensemble.create_programs()
print(f"  {len(ensemble.programs)} partitions created")

In [ ]:
print("Running every partition...")
t0 = time.time()
ensemble.run().join()
elapsed = time.time() - t0

n_parts = len(ensemble.programs)
print(f"Done in {elapsed:.1f}s.")
print(f"  partitions: {n_parts}")
print(f"  total circuits executed: {ensemble.total_circuit_count}")
print("  one QAOA program per community")

solution, energy = ensemble.aggregate_results(
    BeamSearchStrategy(beam_width=3, n_partition_candidates=5)
)
print(
    f"\nGlobal portfolio: {int(solution.sum())} assets selected, QUBO energy = {energy:.6f}"
)

### Small-instance comparison

The same eight-asset QUBO can be solved directly and through community partitioning. Comparing the two makes the aggregation step visible.

In [ ]:
from divi.qprog import BeamSearchStrategy, EarlyStopping
from divi.qprog.problems import CommunityDecomposer
from divi.qprog.workflows import PartitioningProgramEnsemble
import hybrid

small_problem = BinaryOptimizationProblem(
    qubo_matrix,
    decomposer=CommunityDecomposer(max_cluster_size=4, method="modularity", seed=42),
    composer=hybrid.SplatComposer(),
)
small_ensemble = PartitioningProgramEnsemble(
    problem=small_problem,
    quantum_routine="qaoa",
    n_layers=1,
    optimizer=MonteCarloOptimizer(population_size=10, n_best_sets=3),
    max_iterations=3,
    early_stopping=EarlyStopping(patience=2),
    backend=backend,
)
small_ensemble.create_programs()
small_ensemble.run().join()
partitioned_solution, partitioned_energy = small_ensemble.aggregate_results(
    BeamSearchStrategy(beam_width=3, n_partition_candidates=5)
)
direct_energy = bqm.energy(qaoa_solution)
print(f"Direct QAOA energy: {direct_energy:.6f}")
print(f"Partitioned QAOA energy: {partitioned_energy:.6f}")
print(f"Community programs: {len(small_ensemble.programs)}")
for key in small_ensemble.programs:
    print(f"  {key[0]}: {key[1]} variables")

import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.imshow(np.abs(qubo_matrix), cmap="magma")
plt.colorbar(label="absolute QUBO coupling")
plt.title("Small portfolio QUBO interaction matrix")
plt.xlabel("asset index")
plt.ylabel("asset index")
plt.show()

### Local execution

Every partition above runs through `MaestroSimulator`. Adjust the number of assets, the community size, and the optimizer settings to explore the trade-offs in this local example.

### Variation — same workflow, PCE engine instead of QAOA

[**PCE (Pauli Correlation Encoding)**](https://arxiv.org/abs/2401.09421) replaces QAOA's one-binary-variable-per-qubit mapping with a polynomial encoding that uses **logarithmically fewer qubits** for the same number of variables. A 20-asset partition compresses from 20 qubits to ⌈log₂(20+1)⌉ × 2 = 10. Useful when partition sizes still exceed your hardware's qubit count.

The workflow keeps the same `BinaryOptimizationProblem`, `CommunityDecomposer`, and aggregation step. Only the engine arguments change:

```python
from qiskit.circuit.library import CXGate, RYGate, RZGate
from divi.qprog import PCE, GenericLayerAnsatz
from divi.qprog.optimizers import PymooOptimizer, PymooMethod

ensemble = PartitioningProgramEnsemble(
    problem=problem,                         # same community partitioner
    quantum_routine="pce",                   # ← was "qaoa"
    n_layers=3,
    optimizer=PymooOptimizer(PymooMethod.DE, population_size=30),
    max_iterations=15,
    early_stopping=EarlyStopping(patience=5),
    backend=backend,
    # PCE-specific engine kwargs:
    ansatz=GenericLayerAnsatz(
        gate_sequence=[RYGate, RZGate],
        entangler=CXGate,
        entangling_layout="all-to-all",
    ),
    encoding_type="poly",
)
ensemble.create_programs()
ensemble.run().join()
solution, energy = ensemble.aggregate_results(BeamSearchStrategy(beam_width=3, n_partition_candidates=5))
```

Compare the PCE and QAOA formulations while keeping the same community-partitioned problem.

---

## Further exploration

Vary the asset count, risk-return trade-off, and community size, then inspect how the selected assets change.